# F2.2 · Exploración inicial

Se revisa cómo vienen los datos **antes de tocarlos**, para decidir qué hay que limpiar y transformar.

In [1]:
# Configuración común: rutas del proyecto y módulos del pipeline.
import pandas as pd

from pipeline import chilecompra as cc
from pipeline import config, ordenes

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.1f}".format)
print("Raíz del proyecto:", config.PROJECT_ROOT.name)

Raíz del proyecto: 002_sumativa-1-avance-proyecto-f1-y-f2-40


In [2]:
raw = ordenes.load_orders(config.RAW_DATA_DIR)
raw.groupby("periodo").agg(lineas=("Codigo", "size"), ordenes=("Codigo", "nunique"), proveedores=("CodigoProveedor", "nunique"), unidades_de_compra=("CodigoUnidadCompra", "nunique"))

,lineas,ordenes,proveedores,unidades_de_compra
periodo,,,,
2024-09,12025,4598,1476,47
2025-09,13855,5687,1634,53


## 1. Estructura y tipos de datos

Todo se lee como texto para no perder información. Los montos traen **coma decimal**, por lo que no pueden sumarse hasta convertirlos.

In [3]:
print("Tipos al leer:", raw.dtypes.astype(str).value_counts().to_dict())
raw[["Codigo", "FechaEnvio", "MontoTotalOC_PesosChilenos", "TipoMonedaOC", "EsTratoDirecto", "EsCompraAgil"]].head()

Tipos al leer: {'str': 80}


,Codigo,FechaEnvio,MontoTotalOC_PesosChilenos,TipoMonedaOC,EsTratoDirecto,EsCompraAgil
0,2200-524-SE23,2024-09-27,"3401030,71",CLP,Si,No
1,2200-525-SE23,2024-09-26,"426030,71",CLP,Si,No
2,898-2632-SE23,2024-09-13,"19283516,84",CLP,No,No
3,1057554-5174-SE23,2024-09-30,"113595,02",CLP,No,No
4,1057554-5174-SE23,2024-09-30,"113595,02",CLP,No,No


In [4]:
with_comma = raw["MontoTotalOC_PesosChilenos"].str.contains(",").mean()
print(f"Montos escritos con coma decimal: {with_comma:.1%}")
print("Monedas:", raw["TipoMonedaOC"].value_counts().to_dict())

Montos escritos con coma decimal: 43.4%
Monedas: {'CLP': 25848, 'CLF': 23, 'USD': 8, 'UTM': 1}


## 2. Calidad de los datos

La fuente marca los datos ausentes de dos maneras: celda vacía o el texto `NA`. Ninguna de las dos es un NA real para pandas.

In [5]:
missing = raw.apply(lambda column: column.str.strip().isin(ordenes.MISSING_MARKERS))
by_period = missing.groupby(raw["periodo"]).mean().T.mul(100).round(1)
by_period[by_period.max(axis=1) > 5].sort_values("2025-09", ascending=False).head(12)

periodo,2024-09,2025-09
FechaCancelacion,100.0,100.0
Codigo_ConvenioMarco,84.9,85.2
Financiamiento,71.2,74.2
CodigoLicitacion,66.9,67.4
PaisProveedor,38.8,53.8
totalImpuestos,32.9,50.9
EsTratoDirecto,0.0,38.0
ProcedenciaOC,0.0,38.0
CodigoProyectoPlanAnualCompras,100.0,36.7
ActividadComprador,16.6,21.7


In [6]:
print("Filas completamente repetidas:", raw.duplicated().sum())
lines_per_order = raw.groupby("Codigo").size()
print(f"Líneas por orden: mediana {lines_per_order.median():.0f}, máximo {lines_per_order.max()}")
print(f"Órdenes con más de una línea: {(lines_per_order > 1).mean():.1%}")

Filas completamente repetidas: 0
Líneas por orden: mediana 1, máximo 120
Órdenes con más de una línea: 31.8%


## 3. Cambio de definición entre 2024 y 2025

`EsTratoDirecto` no significa lo mismo en los dos años.

In [7]:
pd.crosstab([raw["periodo"], raw["EsCompraAgil"]], raw["EsTratoDirecto"].replace("NA", "(sin dato)"))

EsTratoDirecto        (sin dato)    No    Si
periodo EsCompraAgil                        
2024-09 No                     0  5999  2013
        Si                     0     0  4013
2025-09 No                     0  6799  1750
        Si                  5270     0    36

En 2024 toda compra ágil venía marcada **además** como trato directo. En 2025 las compras ágiles traen ese campo sin dato: la Ley N.° 21.634 separó ambas modalidades. Comparar `EsTratoDirecto` directamente daría una caída falsa del trato directo.

## 4. Hallazgos que orientan la limpieza

| Hallazgo | Consecuencia |
|---|---|
| Ausentes como celda vacía o texto `NA` | Unificarlos como NA reales (notebook 03) |
| Espacios al final de algunos textos | Recortarlos (notebook 03) |
| Montos con coma decimal, leídos como texto | Convertirlos a número (notebook 04) |
| Fechas leídas como texto | Convertirlas a fecha (notebook 04) |
| `EsTratoDirecto` cambia de significado en 2025 | Crear una variable `modalidad` comparable (notebook 04) |
| Varias líneas por orden, con el monto total repetido | Dejar una fila por orden antes de sumar (notebook 04) |
| Sin filas completamente repetidas | No se eliminan filas |